### Reflection and Blogpost Writing

### Setup

In [1]:
llm_config = {"model": "llama3","api_type":"ollama"}

## The task!

In [2]:
task = '''Write a concise but engaging blogpost on the increase in RAM prices. Make sure the blogpost is within 100 words.'''


## Create a writer agent

In [3]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [4]:
reply = writer.generate_reply(messages=[{"content": task ,"role": "user"}])

In [5]:
print(reply)

{'content': '"RAM Rush: The Sudden Spike in Prices"\n\nIn recent months, the price of RAM (Random Access Memory) has seen an unprecedented surge. A mere year ago, 16GB of DDR4 RAM could be had for under $50. Fast forward to today, and you\'re looking at a whopping $100+ for the same amount. What\'s driving this sudden rush? Supply chain disruptions, global demand, and manufacturing costs all play a role. For PC enthusiasts and gamers, this increase means rethinking budgets or waiting it out. But for business users, it may be time to reassess IT infrastructure priorities.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


## Adding reflection 

Create a critic agent to reflect on the work of the writer agent.

In [6]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [7]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):

Write a concise but engaging blogpost on the increase in RAM prices. Make sure the blogpost is within 100 words.

--------------------------------------------------------------------------------
Writer (to Critic):

"RAM Price Hike: What's Behind the Surge?"

In recent months, the cost of Random Access Memory (RAM) has skyrocketed, leaving tech enthusiasts and professionals alike scrambling to adjust their budgets. But what's behind this sudden price hike? A perfect storm of supply chain disruptions, manufacturing issues, and soaring demand for memory-intensive devices like gaming PCs and smartphones is driving up costs. As a result, even modest upgrades can now break the bank. If you're in the market for new RAM, be prepared to dig deeper into your pockets or consider alternative options – the good old days of affordable RAM seem to be behind us.

--------------------------------------------------------------------------------
Critic (to Writer):

**Thought-Provok

## Nested chat

In [8]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)


In [9]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [10]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [11]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

### Orchestrate the nested chats to solve the task

In [12]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [13]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different task.

In [14]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):

Write a concise but engaging blogpost on the increase in RAM prices. Make sure the blogpost is within 100 words.

--------------------------------------------------------------------------------
Writer (to Critic):

**The Ram-Paging Conundrum: Why Your Upgrade Just Got Pricier**

In recent months, the cost of upgrading your laptop or PC's memory has skyrocketed. RAM prices have increased by as much as 30% in some cases, leaving tech enthusiasts and professionals alike scratching their heads. The culprit behind this surge? A combination of factors, including supply chain disruptions, strong demand for high-performance devices, and rising production costs. As the digital world becomes increasingly reliant on memory-intensive applications, it's likely that RAM prices will continue to fluctuate. Upgrade with caution – or consider alternatives like storage optimization.

--------------------------------------------------------------------------------

******************

## Get the summary

In [15]:
print(res.summary)

The Ram-Paging Conundrum: Why Your Upgrade Just Got Pricier

**Meta Title:** Understanding the Rise in RAM Prices: A Guide to Navigating the Market Shift

In recent months, the cost of upgrading your laptop or PC's memory has surged, with some reports indicating a 30% price increase. This sudden hike has left many wondering what's behind this shift and how it will impact their upgrade plans.

**Supply Chain Disruptions: The Root Cause**

The primary culprit behind this surge is a combination of factors, including supply chain disruptions, strong demand for high-performance devices, and rising production costs. The global pandemic has further exacerbated these issues, leading to a shortage of essential components like RAM.

**What This Means for Consumers**

As the digital world becomes increasingly reliant on memory-intensive applications, it's likely that RAM prices will continue to fluctuate. However, this doesn't necessarily mean you should put your upgrade plans on hold. Instead, c